## Cargar Archivos

In [22]:
import os
import re

folder = os.path.join(os.getcwd(), "genes")

# Carga cada archivo de genes/ en una variable con el mismo nombre del archivo
for file in sorted(os.listdir(folder)):
    if file.endswith(".txt"):
        name = re.sub(r"\W", "_", os.path.splitext(file)[0])
        with open(os.path.join(folder, file), encoding="utf-8") as f:
            next(f)  # salta la primera línea (encabezado)
            globals()[name] = "".join(f.read().split())  # sin saltos de línea

gen_M, gen_ORF1AB, gen_S, seq_proteins, SARS_COV_2_MN908947_3, SARS_COV_2_MT106054_1

('...>QHD43415_2AYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGG>QHD43415_3APTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGI

## KMP CON LPS PARA STRING MATCHING

In [23]:
def construir_lps(patron):
    """Tabla de prefijo-sufijo más largo (failure function) del patrón."""
    lps = [0] * len(patron)
    longitud = 0
    i = 1
    while i < len(patron):
        if patron[i] == patron[longitud]:
            longitud += 1
            lps[i] = longitud
            i += 1
        elif longitud:
            longitud = lps[longitud - 1]
        else:
            lps[i] = 0
            i += 1
    return lps


def kmp_buscar(texto, patron):
    """Devuelve los índices (base 0) donde aparece patron en texto."""
    if not patron:
        return []
    lps = construir_lps(patron)
    indices = []
    j = 0
    for i, c in enumerate(texto):
        while j and c != patron[j]:
            j = lps[j - 1]
        if c == patron[j]:
            j += 1
        if j == len(patron):
            indices.append(i - j + 1)
            j = lps[j - 1]
    return indices

In [24]:
# Punto 1: apariciones de cada gen en el genoma MN908947.3 (índices base 0)
genes = {"M": gen_M, "S": gen_S, "ORF1AB": gen_ORF1AB}

for nombre, gen in genes.items():
    indices = kmp_buscar(SARS_COV_2_MN908947_3, gen)
    print(f"Gen {nombre}")
    print(f"  Índices de aparición: {indices}")
    print(f"  Primeros 12 caracteres: {gen[:12]}")

Gen M
  Índices de aparición: [26522]
  Primeros 12 caracteres: ATGGCAGATTCC
Gen S
  Índices de aparición: [21562]
  Primeros 12 caracteres: ATGTTTGTTTTT
Gen ORF1AB
  Índices de aparición: [265]
  Primeros 12 caracteres: ATGGAGAGCCTT


## MANACHER PARA ENCONTRAR PALINDROMOS

In [25]:
def manacher(texto):
    """Devuelve (inicio, longitud) del palíndromo más largo de texto, en O(n)."""
    # Separadores '#' unifican palíndromos de longitud par e impar
    t = "^#" + "#".join(texto) + "#$"
    radios = [0] * len(t)
    centro = derecha = 0
    for i in range(1, len(t) - 1):
        if i < derecha:
            radios[i] = min(derecha - i, radios[2 * centro - i])
        while t[i + radios[i] + 1] == t[i - radios[i] - 1]:
            radios[i] += 1
        if i + radios[i] > derecha:
            centro, derecha = i, i + radios[i]
    longitud, centro = max((r, i) for i, r in enumerate(radios))
    inicio = (centro - longitud) // 2
    return inicio, longitud

In [26]:
# Punto 2: palíndromo más largo de cada gen (se guarda en palindromos.txt)
with open("palindromos.txt", "w", encoding="utf-8") as salida:
    for nombre, gen in genes.items():
        inicio, longitud = manacher(gen)
        palindromo = gen[inicio:inicio + longitud]
        print(f"Gen {nombre}: longitud {longitud} (índice {inicio}) -> {palindromo}")
        salida.write(f"Gen {nombre}\nLongitud: {longitud}\nÍndice: {inicio}\nPalíndromo: {palindromo}\n\n")

Gen M: longitud 11 (índice 493) -> CTAAAGAAATC
Gen S: longitud 15 (índice 2386) -> ATTTTGGTGGTTTTA
Gen ORF1AB: longitud 20 (índice 9711) -> CTCAATGACTTCAGTAACTC


## 3

In [27]:
BASES = "TCAG"
AMINOACIDOS = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
# Tabla del código genético estándar: codón -> aminoácido ('*' = parada)
CODONES = {
    a + b + c: AMINOACIDOS[16 * i + 4 * j + k]
    for i, a in enumerate(BASES)
    for j, b in enumerate(BASES)
    for k, c in enumerate(BASES)
}


def leer_fasta(ruta):
    """Devuelve {id: secuencia} de un archivo FASTA con varias entradas."""
    secuencias = {}
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if linea.startswith(">"):
                actual = linea[1:].split()[0]
                secuencias[actual] = []
            elif linea:
                secuencias[actual].append(linea)
    return {nombre: "".join(partes) for nombre, partes in secuencias.items()}


def traducir(adn, marco=0):
    """Traduce adn a aminoácidos leyendo codones desde la posición marco."""
    return "".join(
        CODONES.get(adn[i:i + 3], "X") for i in range(marco, len(adn) - 2, 3)
    )

In [28]:
# Punto 3: región del genoma que produce cada proteína (índices base 0)
proteinas = leer_fasta(os.path.join(folder, "seq-proteins.txt"))
genoma = SARS_COV_2_MN908947_3
traducciones = [traducir(genoma, marco) for marco in range(3)]


def buscar_proteina(secuencia):
    """Posiciones del genoma donde se codifica secuencia (3 marcos de lectura)."""
    return [
        marco + 3 * i
        for marco in range(3)
        for i in kmp_buscar(traducciones[marco], secuencia)
    ]


def tramos_proteina(secuencia):
    """Lista de (offset_aa, inicio_genoma, n_aa). Si no aparece completa
    (salto de marco ribosomal), se divide en el prefijo más largo y el resto."""
    inicios = buscar_proteina(secuencia)
    if inicios:
        return [(0, inicios[0], len(secuencia))]
    bajo, alto = 1, len(secuencia) - 1
    while bajo < alto:  # prefijo más largo que aparece (monótono -> bin. search)
        medio = (bajo + alto + 1) // 2
        if buscar_proteina(secuencia[:medio]):
            bajo = medio
        else:
            alto = medio - 1
    resto = buscar_proteina(secuencia[bajo:])
    if not buscar_proteina(secuencia[:bajo]) or not resto:
        return []
    return [(0, buscar_proteina(secuencia[:bajo])[0], bajo), (bajo, resto[0], len(secuencia) - bajo)]


for nombre, secuencia in proteinas.items():
    tramos = tramos_proteina(secuencia)
    print(f"Proteína {nombre} ({len(secuencia)} aa)")
    if not tramos:
        print("  No encontrada en el genoma")
        continue
    for offset, inicio, n_aa in tramos:
        fin = inicio + 3 * n_aa - 1
        print(f"  Índices del genoma: {inicio}-{fin}")
    inicio = tramos[0][1]
    codones = [genoma[inicio + 3 * k: inicio + 3 * k + 3] for k in range(4)]
    print(f"  Primeros 4 aminoácidos: {secuencia[:4]}")
    print(f"  Codones: {' '.join(codones)}")
    if len(tramos) > 1:
        print("  (cambio de marco de lectura entre los dos tramos)")

Proteína QHD43415_1 (180 aa)
  Índices del genoma: 265-804
  Primeros 4 aminoácidos: MESL
  Codones: ATG GAG AGC CTT
Proteína QHD43415_2 (638 aa)
  Índices del genoma: 805-2718
  Primeros 4 aminoácidos: AYTR
  Codones: GCA TAC ACT CGC
Proteína QHD43415_3 (1945 aa)
  Índices del genoma: 2719-8553
  Primeros 4 aminoácidos: APTK
  Codones: GCA CCA ACA AAG
Proteína QHD43415_4 (500 aa)
  Índices del genoma: 8554-10053
  Primeros 4 aminoácidos: KIVN
  Codones: AAA ATT GTT AAT
Proteína QHD43415_5 (306 aa)
  Índices del genoma: 10054-10971
  Primeros 4 aminoácidos: SGFR
  Codones: AGT GGT TTT AGA
Proteína QHD43415_6 (290 aa)
  Índices del genoma: 10972-11841
  Primeros 4 aminoácidos: SAVK
  Codones: AGT GCA GTG AAA
Proteína QHD43415_7 (83 aa)
  Índices del genoma: 11842-12090
  Primeros 4 aminoácidos: SKMS
  Codones: TCT AAA ATG TCA
Proteína QHD43415_8 (198 aa)
  Índices del genoma: 12091-12684
  Primeros 4 aminoácidos: AIAS
  Codones: GCT ATA GCC TCA
Proteína QHD43415_9 (113 aa)
  Índices del

In [29]:
# Punto 4: Wuhan 2019 (MN908947.3) vs Texas 2020 (MT106054.1)
wuhan, texas = SARS_COV_2_MN908947_3, SARS_COV_2_MT106054_1
print(f"Longitudes: Wuhan {len(wuhan)}, Texas {len(texas)}")

# Regiones codificantes (inicio, n_aa, proteína) según el punto 3
regiones = [
    (inicio, n_aa, nombre)
    for nombre, secuencia in proteinas.items()
    for _, inicio, n_aa in tramos_proteina(secuencia)
]

n = min(len(wuhan), len(texas))
diferencias = [i for i in range(n) if wuhan[i] != texas[i]]
print(f"Diferencias (comparando las primeras {n} bases): {len(diferencias)}")

for i in diferencias:
    print(f"\nÍndice {i}: Wuhan {wuhan[i]} -> Texas {texas[i]}")
    region = next(
        (r for r in regiones if r[0] <= i < r[0] + 3 * r[1]), None
    )
    if region is None:
        print("  Fuera de las regiones codificantes de las proteínas")
        continue
    inicio, _, nombre = region
    cod = inicio + 3 * ((i - inicio) // 3)
    cw, ct = wuhan[cod:cod + 3], texas[cod:cod + 3]
    aw, at = CODONES[cw], CODONES[ct]
    efecto = "cambia el aminoácido" if aw != at else "mutación silenciosa"
    print(f"  Proteína {nombre}, codón en {cod}-{cod + 2}")
    print(f"  Wuhan: {cw} -> {aw} | Texas: {ct} -> {at}  ({efecto})")


def subcadena_comun_mas_larga(a, b):
    """Longest common substring de a y b en O(|a|+|b|) con un autómata de sufijos de a.
    Devuelve (longitud, fin_en_b) con fin_en_b exclusivo."""
    largo, enlace, trans = [0], [-1], [{}]
    ultimo = 0
    for c in a:
        actual = len(largo)
        largo.append(largo[ultimo] + 1)
        enlace.append(0)
        trans.append({})
        p = ultimo
        while p != -1 and c not in trans[p]:
            trans[p][c] = actual
            p = enlace[p]
        if p != -1:
            q = trans[p][c]
            if largo[p] + 1 == largo[q]:
                enlace[actual] = q
            else:
                clon = len(largo)
                largo.append(largo[p] + 1)
                enlace.append(enlace[q])
                trans.append(dict(trans[q]))
                while p != -1 and trans[p].get(c) == q:
                    trans[p][c] = clon
                    p = enlace[p]
                enlace[q] = enlace[actual] = clon
        ultimo = actual

    estado = longitud = mejor = fin = 0
    for i, c in enumerate(b):
        while estado and c not in trans[estado]:
            estado = enlace[estado]
            longitud = largo[estado]
        if c in trans[estado]:
            estado = trans[estado][c]
            longitud += 1
        if longitud > mejor:
            mejor, fin = longitud, i + 1
    return mejor, fin


longitud, fin = subcadena_comun_mas_larga(wuhan, texas)
comun = texas[fin - longitud:fin]
print(f"\nSubcadena común más larga: {longitud} bases")
print(f"  Índices en Wuhan: {kmp_buscar(wuhan, comun)}")
print(f"  Índices en Texas: {kmp_buscar(texas, comun)}")
print(f"  Primeros 30: {comun[:30]}")

Longitudes: Wuhan 29903, Texas 29882
Diferencias (comparando las primeras 29882 bases): 7

Índice 8781: Wuhan C -> Texas T
  Proteína QHD43415_4, codón en 8779-8781
  Wuhan: AGC -> S | Texas: AGT -> S  (mutación silenciosa)

Índice 18602: Wuhan T -> Texas C
  Proteína QHD43415_13, codón en 18600-18602
  Wuhan: CAT -> H | Texas: CAC -> H  (mutación silenciosa)

Índice 18974: Wuhan T -> Texas A
  Proteína QHD43415_13, codón en 18972-18974
  Wuhan: GTT -> V | Texas: GTA -> V  (mutación silenciosa)

Índice 19174: Wuhan A -> Texas C
  Proteína QHD43415_13, codón en 19173-19175
  Wuhan: GAT -> D | Texas: GCT -> A  (cambia el aminoácido)

Índice 27924: Wuhan C -> Texas T
  Proteína QHD43422, codón en 27923-27925
  Wuhan: ACA -> T | Texas: ATA -> I  (cambia el aminoácido)

Índice 28143: Wuhan T -> Texas C
  Proteína QHD43422, codón en 28142-28144
  Wuhan: TTA -> L | Texas: TCA -> S  (cambia el aminoácido)

Índice 29094: Wuhan C -> Texas T
  Proteína QHD43423, codón en 29092-29094
  Wuhan: TTC 